# 02 -- Baseline Contact Model

**Contact Luck Prototype v0.1**

Trains the Version 0.1 baseline multinomial logistic regression on the cleaned, training-eligible contact events, using the time-based development design (train on 2021-2023, validate on 2024).

**Class weighting note (important):** an earlier version of this baseline trained with `class_weight="balanced"`, which was later found (see notebook 03 and CLAUDE.md) to severely distort predicted probabilities on real data -- rows the model called ~54% likely to be a triple were observed to be a triple ~3.6% of the time. This notebook now trains and reports BOTH variants side by side so the difference is visible directly, and treats the **unweighted** model as the candidate Contact Luck probability baseline. See notebook 03 for the full controlled comparison (plus a naive-prevalence floor benchmark and an optional post-hoc-calibration check).

Prefers the full 2021-2024 development dataset (`cleaned_development_data.parquet`) when available, and falls back to the smaller one-week bootstrap sample (`cleaned_batted_balls.parquet`) -- which will show 0 training rows, since it only covers one week of 2024.

> **2025 is a protected, untouched final-test season. Never tune, iterate, or select features using 2025 results.** This notebook never loads 2025 data.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [1]:
import pandas as pd

from mlb_luck_score.config import CLASS_ORDER, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.models.train_contact_model import (
    evaluate_model,
    predict_proba_ordered,
    train_model,
    validate_probabilities,
)

pd.set_option("display.width", 120)

In [2]:
from mlb_luck_score.config import DEVELOPMENT_SEASONS, PROCESSED_DATA_DIR

DEVELOPMENT_PATH = PROCESSED_DATA_DIR / "cleaned_development_data.parquet"
SAMPLE_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"

if DEVELOPMENT_PATH.exists():
    df = pd.read_parquet(DEVELOPMENT_PATH)
    print(f"Loaded the full development dataset: {len(df)} rows from {DEVELOPMENT_PATH}")
elif SAMPLE_PATH.exists():
    df = pd.read_parquet(SAMPLE_PATH)
    print(
        f"Full development dataset not found at {DEVELOPMENT_PATH}.\n"
        f"Falling back to the one-week bootstrap sample: {len(df)} rows from {SAMPLE_PATH}.\n"
        "Run `make download-development-data` then `make clean-development-data` for the "
        "full 2021-2024 dataset."
    )
else:
    df = None
    print(
        "No cleaned data found. Run one of:\n"
        "  make download-sample && make clean-data                     # small one-week sample\n"
        "  make download-development-data && make clean-development-data # full 2021-2024 dataset\n"
        "then re-run this notebook."
    )

Loaded the full development dataset: 494173 rows from /Users/arihantaneja/Downloads/TrueLuckMLBStat/data/processed/cleaned_development_data.parquet


In [3]:
if df is not None:
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    print(f"{len(training_eligible)} of {len(df)} rows are training-eligible")
else:
    training_eligible = None

486443 of 494173 rows are training-eligible


## Time-based split

Train seasons: configured in `mlb_luck_score.config.TRAIN_SEASONS`. Validation seasons: `mlb_luck_score.config.VALIDATION_SEASONS`. 2025 (`FINAL_TEST_SEASONS`) is never touched here.

In [4]:
if training_eligible is not None:
    train_df = training_eligible[training_eligible["season"].isin(TRAIN_SEASONS)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    print(f"Train seasons {TRAIN_SEASONS}: {len(train_df)} rows total")
    print(f"Validation seasons {VALIDATION_SEASONS}: {len(val_df)} rows total")
else:
    train_df = val_df = None
    print("Skipped -- no data loaded.")

Train seasons (2021, 2022, 2023): 364311 rows total
Validation seasons (2024,): 122132 rows total


### Training rows by season (2021-2023) and validation rows (2024)

In [5]:
if train_df is not None:
    print("Training rows by season:")
    display(train_df.groupby("season").size().rename("training_rows").reindex(list(TRAIN_SEASONS), fill_value=0))
    print("\nValidation rows by season:")
    display(val_df.groupby("season").size().rename("validation_rows").reindex(list(VALIDATION_SEASONS), fill_value=0))
else:
    print("Skipped -- no data loaded.")

Training rows by season:


season
2021    119671
2022    122203
2023    122437
Name: training_rows, dtype: int64


Validation rows by season:


season
2024    122132
Name: validation_rows, dtype: int64

### Outcome counts by split

In [6]:
if train_df is not None:
    train_counts = train_df["outcome_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    val_counts = val_df["outcome_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    outcome_by_split = pd.DataFrame({"train": train_counts, "validation": val_counts})
    display(outcome_by_split)
else:
    print("Skipped -- no data loaded.")

,train,validation
outcome_class,,
out,244612,82451
single,76662,25782
double,24003,7760
triple,2022,695
home_run,17012,5444


### Are all five outcome classes represented?

In [7]:
if train_df is not None:
    train_present = set(train_df["outcome_class"].dropna().unique())
    val_present = set(val_df["outcome_class"].dropna().unique())
    train_missing = set(CLASS_ORDER) - train_present
    val_missing = set(CLASS_ORDER) - val_present

    if train_missing:
        print(f"Training split is MISSING class(es): {sorted(train_missing)}")
    else:
        print("Training split: all five outcome classes represented.")

    if val_missing:
        print(f"Validation split is MISSING class(es): {sorted(val_missing)}")
    else:
        print("Validation split: all five outcome classes represented.")
else:
    print("Skipped -- no data loaded.")

Training split: all five outcome classes represented.
Validation split: all five outcome classes represented.


## Train BOTH variants: unweighted (candidate baseline) vs class-balanced (comparison only)

Same training rows, same validation rows -- `class_weight` is the only thing that differs, isolating its effect.

In [8]:
if train_df is not None and len(train_df) > 0:
    unweighted = train_model(train_df, class_weight=None)
    balanced = train_model(train_df, class_weight="balanced")
    print("Numeric features:", unweighted.numeric_features)
    print("Categorical features:", unweighted.categorical_features)
else:
    unweighted = balanced = None
    print(
        "Skipped -- no training-eligible rows for the configured train seasons. "
        "If you're using the one-week sample, this is expected: run "
        "`make download-development-data` and `make clean-development-data` for "
        "2021-2023 training data."
    )

Training with class_weight='balanced' (class_balanced_comparison_only). This variant is a labeled COMPARISON MODEL ONLY -- its predicted probabilities are known to be severely miscalibrated and must NOT be used for the Contact Luck score.


Numeric features: ['launch_speed', 'launch_angle', 'spray_angle_approx', 'hit_distance_sc']
Categorical features: ['bb_type', 'stand']


## Probability predictions (unweighted candidate baseline)

In [9]:
if unweighted is not None and val_df is not None and len(val_df) > 0:
    feature_cols = unweighted.numeric_features + unweighted.categorical_features
    proba_df = predict_proba_ordered(unweighted, val_df[feature_cols].head(10))
    validate_probabilities(proba_df)
    display(proba_df)
else:
    print("Skipped -- no trained model or no validation rows available.")

,out,single,double,triple,home_run
370193,0.911833,0.079232,0.007684,0.001249,2.651973e-06
370194,0.984836,0.009607,0.005491,0.000063,1.589376e-06
370195,0.871953,0.042978,0.062237,0.009112,1.371919e-02
370196,0.746982,0.025465,0.099321,0.030711,9.752032e-02
370197,0.769179,0.203788,0.024040,0.002993,2.450972e-11
370198,0.861138,0.112624,0.017622,0.008616,1.624394e-12
370199,0.313211,0.331076,0.331846,0.023028,8.382560e-04
370200,0.776200,0.040535,0.092764,0.009803,8.069825e-02
370201,0.435006,0.492116,0.067981,0.004896,8.477281e-07
370202,0.874930,0.075236,0.043550,0.004132,2.151552e-03


## Core evaluation metrics: unweighted vs class-balanced, side by side

In [10]:
if unweighted is not None and val_df is not None and len(val_df) > 0:
    unweighted_metrics = evaluate_model(unweighted, val_df)
    balanced_metrics = evaluate_model(balanced, val_df)

    comparison = pd.DataFrame(
        {
            "unweighted (candidate baseline)": {
                "multiclass_log_loss": unweighted_metrics["multiclass_log_loss"],
                **{f"brier_{c}": unweighted_metrics["brier_score_by_class"][c] for c in CLASS_ORDER},
            },
            "class_balanced (comparison only)": {
                "multiclass_log_loss": balanced_metrics["multiclass_log_loss"],
                **{f"brier_{c}": balanced_metrics["brier_score_by_class"][c] for c in CLASS_ORDER},
            },
        }
    )
    display(comparison)
    print(
        "\nLower is better for both log loss and Brier score. See notebook 03 for "
        "full calibration tables/plots and expected calibration error (ECE) -- log "
        "loss and Brier alone don't show WHERE probabilities are miscalibrated."
    )
else:
    print("Skipped -- no trained model or no validation rows available.")

,unweighted (candidate baseline),class_balanced (comparison only)
multiclass_log_loss,0.670321,1.078047
brier_out,0.165917,0.284009
brier_single,0.137869,0.157697
brier_double,0.052472,0.070634
brier_triple,0.005608,0.041056
brier_home_run,0.017203,0.021588



Lower is better for both log loss and Brier score. See notebook 03 for full calibration tables/plots and expected calibration error (ECE) -- log loss and Brier alone don't show WHERE probabilities are miscalibrated.
